In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
from surfaceomeTopography import *

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
#from Bio.Alphabet import IUPAC

import re

%load_ext autoreload

%autoreload 2

In [ ]:
def select_max(x, col):
    m = x[col].min()
    if len(x) > 1 and (x[col] == m).all():
        return -1
    else:
        return x.loc[x[col].idxmax()].ID
    
    
def create_assignments(df, filter_field = 'disorder', filter_score = False, filter_len = True, drop_len = 1.0, overlap = 4):
    
    field_dict = {'length': filter_field+'_len',
                 'start': filter_field+'_start',
                 'end': filter_field+'_end',
                 }
    
    #drop disorder length of 1, from structures missing amino acids, not useful for disorder calculations
    if filter_len == True:
        df = df[df[field_dict['length']] >= 1.0]

    df_overlaps = find_overlaps(df.copy(), allowed_overlap = overlap, columns = (field_dict['start'],field_dict['end']))
    df_graph = overlap_graph(df_overlaps, grouping_var = 'ID link')
    df_conns = find_connections(df_graph)
    df_data = annotate_subgraphs(df_conns, df_overlaps)

    if filter_score == True:
        
        df_selected = df_data.groupby(['ID link', 'Subgraph'])['score_dom', 'ID'].apply(select_max, col = 'score_dom')
        df_selected = df_selected[df_selected >= 0]
        
    else:
        df_selected = df_data.groupby(['ID link', 'Subgraph'])[field_dict['length'], 'ID'].apply(select_max, col = field_dict['length'])
        df_selected = df_selected[df_selected >= 0]

    df_assigned = df_overlaps.loc[df_overlaps.ID.isin(df_selected), :].drop(columns=['Interval'])
    
    return df_assigned

def find_disorder_str(row, cutoff = None):
    #for alphafold confidence find disorder
    #Delegates to surfaceomeTopography.disorder_height.plddt_disorder_regions.
    #The old inline regex was  '[0-'+str(cutoff)+']{1,}'  with cutoff=5, which is
    #inclusive of the digit 5 and so thresholded at pLDDT<60 while its comment said 50,
    #and which accepted single-residue regions (median region 4 aa, 85% under 20 aa).
    #Threshold now follows Piovesan et al. Protein Sci 2022;31:e4466 (pLDDT<68.8 optimal),
    #with a minimum run length of 20 aa inside the measured span and 5 aa in the
    #confidence-trimmed tails, where nothing else covers the residues.
    from surfaceomeTopography.disorder_height import plddt_disorder_regions
    span = None
    if pd.notna(getattr(row, 'n_trim', None)) and pd.notna(getattr(row, 'c_trim', None)):
        span = (row.ecd_start + row.n_trim, row.ecd_start + row.c_trim)
    #returned in string coordinates; the caller adds ecd_start, so emit ecd_start = 0
    regions = plddt_disorder_regions(row.confidence_string, ecd_start = 0,
                                     measured_span = None if span is None else
                                     (span[0] - row.ecd_start, span[1] - row.ecd_start))
    return(regions)

def largest_unique_ecd(df, groupby_col, selection_col):
    #select for the largest ecd when there are multiple measured ecds
    
    idx = df.groupby([groupby_col])[selection_col].transform(max) == df[selection_col]
    
    return df[idx].drop_duplicates(subset = [groupby_col])

## Download Uniprot Data
##### For Each Proteome of Interest (human and mouse)

proteomes are specified in proteomes.tab contained within data_path/proteomes/
each file type is downloaded to its respective folder (proteomes/gff, etc.)
only needs to be run once, running again may overwrite previous proteome version and change subsetting results

#### Load Human Proteome Data
for domain subsetting

In [ ]:
proteome = 'UP000005640'

#routed through resolve() so SURFACEOME_REFRESH can supply a newer proteome pull;
#with the overlay unset these are the same archive files as before.
gff = import_gff(resolve(f'inputs/proteome/{proteome}.gff'))
tab = import_tab(resolve(f'inputs/proteome/{proteome}.tab'))
fasta = import_fasta(resolve(f'inputs/proteome/{proteome}.fasta'))

## Generate dataframes with ecds, GPI anchored ecds, icds, and tmds using annotations from tab and gff proteome files

In [ ]:
#subset ecds and icds
gpi = subset_gpi(tab,gff)
ecds = subset_ecds(gff, tab)
icds = subset_icds(gff)
tmds = subset_tmd(gff)

#subset icds and TM domains to surfaceome only
icds = icds[icds['ID link'].isin(ecds['ID link'])|gpi['ID link'].isin(gpi['ID link'])]

tmds = tmds[tmds['ID link'].isin(ecds['ID link'])|gpi['ID link'].isin(gpi['ID link'])]

#how many ecds identified?
ecds['ID link'].describe()

In [ ]:
#import surfaceome mapping to gene names
names = pd.read_csv(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab'),
                      sep = '\t')


#names missing from above, downloading individually
names_cleanup = pd.read_csv(resolve('inputs/mapping/uniprot_mapping_cleanup.tab'),  index_col=None,
                      sep = '\t')

names = pd.concat([names, names_cleanup])

#explode id link columns so that there are singel assignments
names['ID link'] = names.apply(lambda row: row['ID link'].split(','), axis = 1)
names = names.explode('ID link')

#uniprot_mapping_cleanup.tab supplies two accessions the 2022 export missed (P0DTE4,
#P31358).  A refreshed export already covers them, so the concat above then holds each
#twice and the left merge at the structures step duplicates that protein's row -- CD52
#appeared twice in the 2026 table.  No-op on the 2022 mapping, which has neither.
names = names.drop_duplicates(subset = ['ID link'], keep = 'first')

#One entry per gene: drop unreviewed (TrEMBL) entries -- fragments and isoform-like sequences --
#whose gene has a reviewed (Swiss-Prot) entry. Otherwise each becomes an ectodomain with its own
#height, and in the cell-cell panels takes its gene's full expression. The 2026 proteome carries
#many more of them than the 2022 one did: 66 in the surfaceome against 8.
gene = names.set_index('ID link')['Gene names  (primary )']
status = names.set_index('ID link')['Status']
duplicate_entries = set(status.index[(status == 'unreviewed') & gene.isin(set(gene[status == 'reviewed'].dropna()))])
ecds, gpi, icds, tmds = (d[~d['ID link'].isin(duplicate_entries)] for d in (ecds, gpi, icds, tmds))
len(duplicate_entries)

## Write out  fasta sequences for all ecds as well as the subset that can be modeled with phyre2

In [ ]:
#id all ecds
all_ecd_seq = fasta_subset(ecds, fasta)
all_gpi_seq = fasta_subset(gpi, fasta)


#save gpi data

#save dataframe containing all surface protein info
total = surfaceome_df(gpi, ecds)
total.to_csv(out_path('database/'+proteome+'_ecds_total.csv'),
            index = False)

total['ID link'].describe()


In [ ]:
# Phyre2 input FASTA files are no longer written here; stage 0 (build_surfaceome.py) writes the
# ectodomain FASTA that hmmsearch runs over. The Phyre2 length filter is kept for the record.
#filter sequence lengths allowed in phyre2 modeling (>30, <3000)
phyre_ecd = phyre_filter(ecds)
phyre_gpi = phyre_filter(gpi)

#get sequences for phyre filtered ecds
phyre_gpi_seq = fasta_subset(phyre_gpi, fasta)
phyre_ecd_seq = fasta_subset(phyre_ecd, fasta)


#get short subset
short = ecds[(ecds['seq_len'] < 30)]
short_seq = fasta_subset(short, fasta)





## ECDs sent to phyre2 for modeling

- Input: fasta sequence files (max 500 seqs)
- Output: pdb files for best model, and summary stats for each model

Followed by measurement of pdb model outputs with:
*OpenAndMeasureSize_auto.py*
- Input: file directory with phyre2 output folders
- Output: file with measurements for each ecd model
    - NOTE: some models have bad atoms that are unable to run in the measurement script, these are skipped, leaving no measurement in the file.


## Order the Domains for Each Protein: ECD Classifications
Here I am ordering each domain including transmembrane domains in order to identify **multipass** proteins, classify extracellular **loops**, and id those ecds with **exposed N-termini**

In [ ]:
#order each domain
#some proteins have multiple chain annotations if they start on the same amino acid they are given the same rank
ordered = order_domains(ecds, icds, tmds, gpi)

#count the transmembrane domains within each protein
tm_count = ordered[ordered['domain']=='tmd'].groupby('ID link')['domain'].count().reset_index().rename(columns ={'domain': 'tmd_count'})

ordered_domains = ordered.merge(tm_count, on = 'ID link', how = 'outer')

#assign multipass
ordered_domains = ordered_domains.assign(multipass = np.where(ordered_domains.tmd_count > 1, 'multipass', np.where(ordered_domains.source == 'gpi', 'gpi-anchor', 'singlepass')))

#assign exposed N-termini
first_domains = ordered_domains[['ID link', 'domain_order', 'source']][ordered_domains.domain_order == 1]
exposed_N = first_domains.assign(N_termini = np.where(first_domains.source == 'gpi', 'exposed', np.where((first_domains.source == 'ecd'), 'exposed', 'cytoplasmic')))

ordered_domains = ordered_domains.merge(exposed_N[['ID link', 'N_termini']], on = 'ID link', how = 'outer')

#assign loops
domain_order = ordered_domains[['ID link','domain_order']]
domain_order_max = domain_order.sort_values(['ID link','domain_order']).groupby('ID link').max().reset_index().rename(columns ={'domain_order': 'domain_order_max'})
domain_order_min = domain_order.sort_values(['ID link','domain_order']).groupby('ID link').min().reset_index().rename(columns ={'domain_order': 'domain_order_min'})
ecds_tmds = domain_order.merge(domain_order_max, on = ['ID link'], how = 'outer')
ecds_tmds = ecds_tmds.merge(domain_order_min, on = ['ID link'], how = 'outer')

ecds_labeled = ordered_domains[ordered_domains['source'].str.contains('ecd')][['ID link','domain_order', 'source']]
ecd_order = ecds_labeled.merge(ecds_tmds, on = ['ID link', 'domain_order'],  how = 'outer')
ecd_loops = ecd_order.assign(ecd_domain_order = np.where(ecd_order.domain_order == ecd_order.domain_order_min, 'first', np.where(ecd_order.domain_order == ecd_order.domain_order_max, 'last', 'loop')))

domains_labeled = ordered_domains.merge(ecd_loops[['ID link','domain_order', 'ecd_domain_order']], on = ['ID link', 'domain_order'] , how = 'outer')

#assign short, modeled, long protiens
ecds_labeled = domains_labeled[domains_labeled['source'].str.contains('ecd|gpi')]

ecds_classified = ecds_labeled.assign(phyre_eligible = np.where(ecds_labeled.seq_len <= 30, 'short', np.where(ecds_labeled.seq_len >=3000, 'long', 'modeled')))

ecds_classified.to_csv(out_path('database/'+proteome+'_surfaceome_ecds_classified.csv'),
            index = False)

largest_ecds = largest_unique_ecd(ecds_classified, 'ID link', 'seq_len')

ecds_classified['ID link'].describe()

In [ ]:
largest_ecds.to_csv(out_path('database/'+proteome+'_surfaceome_largest_ecds_classified.csv'),
            index = False)

## Disorder Labeling

### retrivial of disorder annotations for each protein:
disorder annotions retrieved from mobidb database: <https://mobidb.bio.unipd.it/>

In [ ]:
#surfaceome accessions
accessions = ecds_classified['ID link'].unique().tolist()

#through resolve(), so a refreshed MobiDB in the overlay is picked up;
#import_disorder() appends <proteome>_disorder_<method>.txt to this prefix
output_disorder = str(resolve('inputs/disorder/'+proteome+'_disorder_full.txt'))\
                    .replace(proteome+'_disorder_full.txt', '')

#data retrival from mobidb database, saved to file
#retrieve_disorder(accessions, proteome, output_disorder)

#import and reformat  disorder annotation file
disorder = import_disorder(output_disorder, proteome, method = 'full')
disorder_lite = import_disorder(output_disorder, proteome, method = 'lite')

disorder = parse_disorder(disorder)
disorder_lite = parse_disorder(disorder_lite)

#combine disorder annotations
disorder = pd.concat([disorder, disorder_lite])

#isolate extracellular disorder
ecd_disorder = surface_disorder(disorder, surfaceome = ecds_classified[['ID link', 'start', 'end']])

ecd_disorder = ecd_disorder.drop_duplicates(subset = ['ID link','start', 'end','disorder_start', 'disorder_end'])

#resolve overlaps picking the longest disorder annotations when a clash is encounterd
disorder_assigned = create_assignments(ecd_disorder, filter_field = 'disorder', drop_len = 1.0, overlap = 4)

disorder_within_largest_ecds = pd.merge(disorder_assigned, largest_ecds, on = ['ID link', 'start', 'end'])

#summarize disorder labeling accross all ecds
surfaceome_disorder = summarize_disorder(disorder_assigned, ecds_classified)

surfaceome_disorder['percent_disorder'].describe()

In [ ]:
longest_seq_disorder = surfaceome_disorder.sort_values('seq_len', ascending = False).drop_duplicates('ID link')

In [ ]:
longest_seq_disorder.to_csv(out_path('database/'+proteome+'_surfaceome_largest_ecds_domain_counting_disorder.csv'), index = False)

In [ ]:
surfaceome_disorder.to_csv(out_path('database/'+proteome+'_surfaceome_disorder_domain_counting_disorder.csv'), index = False)

In [ ]:
disorder_within_largest_ecds.to_csv(out_path('database/'+proteome+'_disorder_within_largest_ecd_domain_counting_only.csv'), index = False)

# Extract Disorder information from AlphaFold
alphafold incorporates confidence metrics, low confidence corresponds to disordered regions

## Merge/Resolve Disorder information with previous mobiDB annotations
account for overlaping annotations

In [ ]:
af_confidence = pd.read_csv(resolve('inputs/alphafold/af_confidence.csv'))
#one entry per gene: drop the duplicate entries named in the gene-name cell
af_confidence = af_confidence[~af_confidence['ID'].isin(duplicate_entries)]

#set cutoff for disorder calling...alphafold documenation indicates 50 (i.e. 5)as the disorder cutof
af_confidence = af_confidence.assign(af_disorder = af_confidence.apply(lambda row: find_disorder_str(row, cutoff =5), axis = 1))

#make these annotations look like the previous disorder annotations
af_explode = af_confidence.explode('af_disorder').reset_index().drop(columns = ['index'])

##resolve alphafold/uniprot id mappings
af_mappings =  pd.read_csv(resolve('inputs/proteome/surfaceome_current.tsv'), sep = '\t', header = None, 
                           names = ['current', 'af_id'])

af_mappings['current'] = af_mappings.apply(lambda row: row.current.split(','), axis = 1)
af_mappings = af_mappings.explode('current')
af_mappings = af_mappings[~af_mappings['current'].isin(duplicate_entries)]

#make these annotations look like the previous disorder annotations
af_disorder = pd.DataFrame(af_explode['af_disorder'].values.tolist(), columns=['disorder_start','disorder_end'])
af_disorder = pd.merge(af_explode, af_disorder, left_index = True, right_index = True)
af_disorder = af_disorder.assign(disorder_len = af_disorder.disorder_end-af_disorder.disorder_start)
#af_disorder = af_disorder.rename(columns = {#'ID': 'ID link',
 #                                          'ecd_start': 'start',
  #                                         'ecd_end': 'end'})

af_disorder['disorder_start'] =af_disorder['disorder_start'] + af_disorder['ecd_start']
af_disorder['disorder_end'] =af_disorder['disorder_end'] + af_disorder['ecd_start']

largest_ecds_mapped = pd.merge(largest_ecds, af_mappings, left_on = 'ID link', right_on = 'current')

#save previous uniprot annotations and current alphafold mapping
largest_ecds_mapped.to_csv(out_path('tables/largest_unique_ecds_af_mappings.csv'))

af_ecds_disorder = pd.merge(af_disorder, largest_ecds_mapped, left_on = ['ID'], right_on = ['af_id'])

#now have disorder annotations resolved from alphafold
af_ecds_disorder['ID link'].describe()

#give alpha fold disorder annotations a higher rank
#see bioriv paper that says AF out performs other disorder predictions

af_ecd_disorder= af_ecds_disorder[~((af_ecds_disorder['disorder_len'] <= 1.0)|af_ecds_disorder['disorder_len'].isnull())]

#af_ecd_disorder = af_ecd_disorder.assign(score_dom = 10)

#disorder_within_largest_ecds = disorder_within_largest_ecds.assign(score_dom = 1)

af_mobidb_disorder = pd.concat([disorder_within_largest_ecds, af_ecd_disorder]).reset_index().drop(columns = 'ID')

af_disorder_assigned =  create_assignments(af_mobidb_disorder, filter_field = 'disorder', drop_len = 1.0, overlap = 4)

af_disorder_assigned.sort_values('disorder_len')

## Identification of Glycosylation Sites

utilizing glycosite annotations in the uniprot database to identify glycosites in the surfaceome

In [ ]:
glycosites = parse_glycosites(gff, accessions)

ecd_glycosites = surface_glycosites(glycosites, ecds_classified)

ecd_glycosites.to_csv(out_path('database/glycosites.csv'), index = False)

#incorporate glycomine annotations. annotations downloaded from:
#https://glycomine.erc.monash.edu/Lab/GlycoMine/#webserver
n_glycans  = pd.read_csv(resolve('inputs/glycosylation/glycomine/N_result.txt'), sep = '\s')
o_glycans  = pd.read_csv(resolve('inputs/glycosylation/glycomine/O_result.txt'), sep = '\s')
c_glycans  = pd.read_csv(resolve('inputs/glycosylation/glycomine/C_result.txt'), sep = '\s')

n_glycans = n_glycans.assign(glycan_type = 'N')
o_glycans = o_glycans.assign(glycan_type = 'O')
c_glycans = c_glycans.assign(glycan_type = 'C')

#only used glycans above threshold (sugested from glycomine)
n_glycans = n_glycans[n_glycans['Value'] > 0.500]
o_glycans = o_glycans[o_glycans['Value'] > 0.502]
c_glycans = c_glycans[c_glycans['Value'] > 0.555]

glycoMine = pd.concat([n_glycans, o_glycans, c_glycans])

glycomine = parse_glycoMine(glycoMine, accessions)

ecd_glycomine = surface_glycosites(glycomine, ecds_classified)

#merge/resolve overlaps

merged_glycans = pd.merge(ecd_glycomine, ecd_glycosites, on = ['ID link', 'glycosite', 'start', 'end'], how = 'outer')

#Union in GlyGen experimental glycosylation sites when the refresh overlay provides them.
#GlycoMine (2014) recovers 51% of experimentally known sites; GlyGen in turn badly
#under-counts dense mucin VNTRs, where individual glycans are never mapped. Neither is
#right alone -- see surfaceomeTopography/download_glygen.py. The union adds ~5% to total
#disorder height and fixes cases like PSGL-1 that GlycoMine calls unglycosylated.
_glygen_path = resolve('inputs/glycosylation/glygen_sites_human.csv')
if _glygen_path.is_file():
    _gg = pd.read_csv(_glygen_path)
    _gg = _gg[_gg['acc'].isin(accessions)][['acc', 'site']].drop_duplicates()
    _gg = _gg.rename(columns = {'acc': 'ID link', 'site': 'glycosite'})
    _gg = surface_glycosites(_gg, ecds_classified)
    merged_glycans = pd.concat([merged_glycans, _gg], sort = False).drop_duplicates(
        subset = ['ID link', 'glycosite', 'start', 'end'], keep = 'first')
    print(f'glycosites after GlyGen union: {len(merged_glycans):,}')
else:
    print('GlyGen sites not present; using GlycoMine + UniProt only')

merged_glycans = merged_glycans.assign(glycan_type = np.where(merged_glycans.note.str.contains('N-linked'), 'N', np.where(merged_glycans.note.str.contains('O-linked'), 'O', np.where(merged_glycans.note.str.contains('C-linked'), 'C', merged_glycans.glycan_type))))

merged_glycans.drop(columns = ['Value', 'td'], inplace = True)

merged_glycans['ID link'].describe()

## Glycosylation and Disorder Overlap
1. check for glycosites in disordered regions

    a. are all disordered regions glycosylated?
    
    b. can we use a bottle brush estimate height for all disordered regions?
    
    c. or are there regions that are better served by another estimate method?

In [ ]:
disorder_gly = af_disorder_assigned[['ID link', 'start', 'end', 'disorder_start', 'disorder_end']].merge(merged_glycans, on =['ID link', 'start', 'end'], how ='left')

disorder_glycosites = disorder_gly[
                    (disorder_gly['glycosite'] <= disorder_gly['disorder_end'])
                    &(disorder_gly['glycosite'] >= disorder_gly['disorder_start'])].drop_duplicates(subset = [
    'disorder_start','disorder_end', 'glycosite' ], keep = 'first')

#unglycosylated disorder
ungly_disorder = disorder_gly[disorder_gly['glycosite'].isnull()]

#combine datasets
disorder_glycosites = pd.concat([disorder_glycosites, ungly_disorder])
#drop duplicate annotations
found_disorder = disorder_glycosites[['ID link','start','end','disorder_start','disorder_end']].drop_duplicates()

#disorder annotations for all glycosites and disorder
total_disorder = pd.merge(af_disorder_assigned[['ID link', 'start', 'end', 'disorder_start', 'disorder_end']], found_disorder, on = ['ID link', 'start', 'end', 'disorder_start', 'disorder_end'], how = 'outer', indicator = True)

#disorder annotations without gly
not_found_gly = total_disorder[total_disorder._merge == 'left_only'].drop(columns = ['_merge'])

disorder_glycosites = pd.concat([disorder_glycosites, not_found_gly])

#get glycosylation counts
gly_count = disorder_glycosites.groupby(['ID link', 'disorder_start', 'disorder_end'])['glycosite'].count().reset_index()

gly_count.rename(columns = {'glycosite': 'glycan_count'}, inplace = True)

disorder_gly_count = total_disorder[['ID link','start', 'end','disorder_start', 'disorder_end']].drop_duplicates(keep = 'first').merge(gly_count, on = ['ID link', 'disorder_start', 'disorder_end'], how = 'left')

disorder_gly_count = disorder_gly_count.assign(disorder_len = disorder_gly_count.disorder_end-disorder_gly_count.disorder_start)
disorder_gly_count = disorder_gly_count.assign(glycan_density = disorder_gly_count.glycan_count/disorder_gly_count.disorder_len)
disorder_gly_count = disorder_gly_count.fillna(0)

#probably want to group by ecd domain, then sum, disorder len glycan count, mean std of glycan density, 
#disorder_gly_count.head(20)

#ecds_classified

disorder_classified = pd.merge(disorder_gly_count, ecds_classified, on = ['ID link', 'start', 'end'], how = 'left')

#disorder_classified[disorder_classified.glycan_density < 0.02]

disorder_classified['ID link'].describe()

In [ ]:
import plotly.express as px
fig = px.histogram(disorder_gly_count, x="glycan_density")
fig.show()

In [ ]:
fig = px.scatter(disorder_gly_count, x="glycan_density", y="disorder_len", color = 'glycan_count', 
                 log_y = 'True', range_color=[1,50], hover_data = ['ID link'])
fig.show()

#are these protiens without glycosylated disorder mostly multipass?
#do some of these disorder annotations give way to domains when the datasets are merged?

## Domain Annoations: Identification and Assignment
hmmer domain identification and assignment of best domains

hmmsearch --tblout dom_all.out Path/Pfam-A.hmm '**/surfaceome_modeling/data/databases/proteomes/output/proteomesUP000005640_all_ecds.fasta'

## Domain assigment for Human Extracellular domains

Using the domain level output from the hmmscan of the Pfam-A hmm profiles, assign each extracellular domain, non overlapping domains.

Steps:
1. Find best domain assignment for each ECD
2. pick that domain
3. find next best, check for overlap
4. if no overlap keep, if overlap detected discard

commented out as already performed and loading from previous run

## Merge Domain and Disorder assignments
#### resolve any overlaps: 

In [ ]:
#set target name for merging with domain assignments
#create ecd ID
disorder_classified = disorder_classified.assign(target_name = disorder_classified['ID link'] + '[' + 
                   disorder_classified['start'].map(int).map(str) + ':' +
                   disorder_classified['end'].map(int).map(str) +']')

domain_assigned = pd.read_csv(resolve('inputs/domains/domain_assignments.csv'))
domain_assigned.drop(columns = ['Unnamed: 0'], inplace = True)
#one entry per gene: drop the duplicate entries named in the gene-name cell
domain_assigned = domain_assigned[~domain_assigned['target_name'].str.split('[').str[0].isin(duplicate_entries)]

#combine disordered and structured domain annoations
comb = pd.concat([domain_assigned, disorder_classified], sort = False)


#set domain start and end for both structured and disorded domains
comb['dom_start'] = comb.apply(lambda row: row['disorder_start'] if (np.isnan(row['dom_start']) == True) else row['dom_start'], axis = 1)
comb['dom_end'] = comb.apply(lambda row: row['disorder_end'] if (np.isnan(row['dom_end']) == True) else row['dom_end'], axis = 1)
comb = comb.assign(dom_len = comb.dom_end - comb.dom_start)

#fill missing domain scores so that ordered domains are selected over disordered domains
comb = comb.fillna(value = {'score_dom': comb['score_dom'].min()-1})

doms_disorder = create_assignments(comb, filter_field = 'dom', drop_len = 1.0, overlap = 4, filter_score = True)

doms_disorder.to_csv(out_path('tables/domain_disorder_annotations.csv'))

## Estimation of protien size following domain and disorder annotation

These methods should be combined to find the best possible height estimates for the largest number of proteins

priority:
1. homology modeling (well modeled) (incl. alphafold)
2. homology modeling + domain/disorder annotation (high confidence model, but poor coverage)
3. domain/disorder annotation (no high confidence models/or disorder annotion)

see what is left and then try and address any issues

issues:
plexin height, min_dimdomain_sizes = pd.read_csv(resolve('code/raw_code/data/human_domain_sizes_oriented_clans_20200123.csv'))

In [ ]:
#import domain and disorder annotations
#doms_disorder = pd.read_csv(resolve('inputs/domains/domain_disorder_annotations_20210419.csv'))

#import domain sizes
domain_sizes = pd.read_csv(resolve('inputs/domains/domain_clan_heights.csv'))

#import homology modeling size estimations
homology = pd.read_csv(resolve('inputs/phyre2/surfaceome_all_ecd_sizes_disorder_extended_UP000005640.csv'))
#one entry per gene: drop the duplicate entries named in the gene-name cell
homology = homology[~homology['ID link'].isin(duplicate_entries)]

#import homology model check
valid = pd.read_csv(resolve('inputs/structures/pdb_check_2020.txt'), 
                    header = None, names = ['Job id', 'valid', 'max_dist', 'count'], sep = '\t')

#conform job id name
valid.loc[:,'Job id']= valid['Job id'].apply(lambda x : x.split('.')[0])

# #import surfaceome mapping to gene names
# names = pd.read_csv(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab'),
#                       sep = '\t')

# #explode id link columns so that there are singel assignments
# names['ID link'] = names.apply(lambda row: row['ID link'].split(','), axis = 1)
# names = names.explode('ID link')

#import surfaceome annotations
surfaceome = ecds_classified

In [ ]:
#integrate domain sizes and domain/disorder annotations
#match domain annotations to measured domain pfam
heights = pd.merge(doms_disorder, domain_sizes[['Pfam', 'height', 'clan_name']].drop_duplicates(), on = ['Pfam', 'height'],
          how = 'left')

#calculate domain lenghts from envelops rather than alignment start/end given poor alignment on the edges/
heights = heights.assign(dom_len = np.where(heights.env_to.notnull(), (heights.env_to-heights.env_from), heights.dom_len))

#create disorderd domain size estimate

#Disordered-domain heights.
#Replaces the binary switch (PL_gly=4.0 / PL_idp=0.5 at glycan_cutoff=0.03) with a
#persistence length that varies continuously with glycan density, merges disordered rows
#that are contiguous with nothing structural between them into a single polymer, and lets
#l_p vary along a merged run via the exact heterogeneous worm-like chain.
#See code/surfaceomeTopography/README_disorder_height.md for the argument and the numbers.
from surfaceomeTopography.disorder_height import assign_disorder_heights

heights = assign_disorder_heights(heights, id_col = 'target_name')

#check for unmeasured domains
heights[np.isnan(heights['height'])]
#heights contains domain heights for each ecd

#cleanup ID column
heights.drop(columns = 'ID', inplace = True)

#create size and coverage estimates using domain counting only

heights_total = heights.groupby('target_name').agg(   
    {  
        'ID link': 'first',
        'height':['count','sum'],    # Sum height
         'disorder_len': ['count','sum'], #length of disordered region
         'dom_len': ['count','sum'], #length of domain annotation
        'qlen': ['count','sum'], #length of the query sequence
         'seq_len': 'first',# length of extracellular domain
        'glycan_density' : ['count', 'mean', 'min', 'max']
        
    })

heights_total.columns = ["_".join(x) for x in heights_total.columns.ravel()]
heights_total.reset_index(inplace = True)

#caluclate protein coverage of the annotations!
#dom_len already carries the disorder lengths (dom_len == disorder_len on every disorder
#row), so adding disorder_len_sum again counted those residues twice: 39% of ectodomains
#reported >100% coverage, CD43 195.7%, max 3488. best_coverage gates the 'well modelled'
#classification at >=70, so the inflation let 70 ectodomains through that should not pass.
heights_total['dom_coverage'] = (heights_total['dom_len_sum'])/heights_total['seq_len_first']

#prep dataframes for merge/concat, reduce number of columns
#need to concat, so column names need to be consistant
heights.rename(columns = {'target_name': 'ID'}, inplace = True)

Prep homology measurements dataframe to merge with domain counting heights

In [ ]:
#create combined datafraome with domain estimates from both domain/disorder estimation and homology_filtered modeling
homology_filtered = pd.merge(homology, valid, on = 'Job id', how = 'outer')

homology_filtered = homology_filtered.dropna(subset = ['ID link','Job id'])

homology_filtered = homology_filtered.assign(ID = homology_filtered['ID link'] + '[' + 
                   homology_filtered['start'].map(int).map(str) + ':' +
                   homology_filtered['end'].map(int).map(str) +']')

#drop unused columns
homology_filtered.drop(columns = ['disorder_first','disorder_last','disorder_count','disorder_coverage',
             'disorder_density','ecd_counts','multipass','percent_disorder', 'note', 'td', 'x', 'y', 'z', 'disorder_len'],
             inplace = True)

#assign domain start/end for homology_filtered model
homology_filtered = homology_filtered.assign(dom_start = homology_filtered['Qstart']+homology_filtered['start'],
                          dom_end = homology_filtered['Qend']+homology_filtered['start'])
#assign domain length
homology_filtered = homology_filtered.assign(dom_len = homology_filtered.dom_end-homology_filtered.dom_start)
homology_filtered = homology_filtered.assign(homology_len = homology_filtered.dom_end-homology_filtered.dom_start)

#assign homology_filtered length
homology_filtered = homology_filtered.assign(homology_filtered_len = homology_filtered.dom_end-homology_filtered.dom_start)

In [ ]:
#combine homology_filtered and domain annotated dataframes to incoorporate all data
dom_plus = pd.concat([heights, homology_filtered], sort = False)

#insure that all proteins have an id assignment (short and long prots are missing)
dom_plus = dom_plus.assign(ID = dom_plus['ID link'] + '[' + 
                   dom_plus['start'].map(int).map(str) + ':' +
                   dom_plus['end'].map(int).map(str) +']')

# #assign low score of -1.0 to short proteins, low prescidence in selecting domains 
# dom_plus = dom_plus.assign(score_dom = dom_plus.apply(
#     lambda row: -1.0 if ((row['source']) == 'short') else row['score_dom'],
#     axis = 1
# ))

# #assign start and end of domain as ecd start/end for short proteins
# dom_plus = dom_plus.assign(dom_start = dom_plus.apply(
#     lambda row: row['start'] if ((row['source']) == 'short') else row['dom_start'],
#     axis = 1),
#     dom_end = dom_plus.apply(
#     lambda row: row['end'] if ((row['source']) == 'short') else row['dom_end'],
#     axis = 1))

#set score as max domain score for well modeled proteins
#will select homology models over domains in those cases
dom_plus = dom_plus.assign(score_dom = dom_plus.apply(
    lambda row: (dom_plus['score_dom'].max()+1) if (row['Confidence (%)'] >= 80) else row['score_dom'],
    axis = 1
))

#assign low score to homology models that have a confidence less than 80%
dom_plus = dom_plus.assign(score_dom = dom_plus.apply(
    lambda row: (dom_plus['score_dom'].min()-1) if (row['Confidence (%)'] <= 80)|(row['valid'] == False) else row['score_dom'],
    axis = 1
))

#find the overlaps between domains and homology
#filter out homology domains that are not valid or if the domain start/end do not conform
#since I filter valid here, no need to filter again later
dom_plus_len = dom_plus[~(dom_plus.valid == False)&~(dom_plus['Confidence (%)'] < 80) & (dom_plus['dom_start'] <= dom_plus['dom_end'])].copy()

homology_assigned = create_assignments(dom_plus_len, filter_field = 'dom', filter_score = True, drop_len = 1.0, overlap = 4)

homology_assigned['ID link'].describe()

# homology_overlaps = find_overlaps(dom_plus_len, allowed_overlap = 4, columns = ('dom_start','dom_end'))
# homology_graph = overlap_graph(homology_overlaps, grouping_var = 'ID link')
# homology_conns = find_connections(homology_graph)
# homology_data = annotate_subgraphs(homology_conns, homology_overlaps)

# homology_selected = homology_data.groupby(['ID link', 'Subgraph'])['score_dom', 'ID'].apply(select_max, col = 'score_dom')#domain_overlap.select_max, col = 'score_dom')
# homology_selected = homology_selected[homology_selected >= 0]

# homology_assigned = homology_overlaps.loc[homology_overlaps.ID.isin(homology_selected), :].drop(columns=['ID','Interval'])

# Import and cleanup alphafold measurments

In [ ]:
#import homology modeling size estimations
alphafold = pd.read_csv(resolve('inputs/alphafold/alphafold_dimensions.txt'), sep = '\t', names = ['ID', 'x', 'y', 'z'])
#one entry per gene: drop the duplicate entries named in the gene-name cell
alphafold = alphafold[~alphafold['ID'].isin(duplicate_entries)]

alphafold_measured = pd.merge(af_confidence, alphafold, on = 'ID', how = 'outer')
alphafold_measured = find_dims(alphafold_measured)

alphafold_measured = pd.merge(alphafold_measured, af_mappings, left_on = 'ID', right_on = 'af_id')
alphafold_measured.rename(columns={'current': 'ID link', 'ID': 'af_ID', 'ecd_start': 'start', 'ecd_end': 'end'}, inplace = True)

#create combined dataframe with domain estimates from both domain/disorder estimation and alphafold modeling

#limit to those with a valid model
alphafold_measured = alphafold_measured.dropna(subset = ['ID link','mean_confidence_trimmed'])

alphafold_measured = alphafold_measured.assign(ID = alphafold_measured['ID link'] + '[' + 
                   alphafold_measured['start'].map(int).map(str) + ':' +
                   alphafold_measured['end'].map(int).map(str) +']')

#drop unused columns
alphafold_measured.drop(columns = ['af_disorder', 'af_id'],
              inplace = True)

#assign domain start/end for alphafold model
alphafold_measured = alphafold_measured.assign(dom_start =  alphafold_measured['n_trim']+ alphafold_measured['start'],
                          dom_end =  alphafold_measured['c_trim']+ alphafold_measured['start'])
#assign domain length
alphafold_measured =  alphafold_measured.assign(dom_len = alphafold_measured.dom_end-alphafold_measured.dom_start)

#assign homology length
alphafold_measured = alphafold_measured.assign(alphafold_len = alphafold_measured.dom_end-alphafold_measured.dom_start)

#assgin ID

#A solved structure is a measured segment of the ectodomain, so it enters the assignment the
#same way an AlphaFold model does rather than replacing the protein's height wholesale.  It
#covers residues SP_BEG..SP_END; whatever it does not cover keeps its domain, disorder or
#sequence contribution, and the overlap resolution decides between them exactly as it does for
#a model.  Knowing the structure's first and last residue is what makes that possible, which is
#why the span is clipped to the ectodomain and carried explicitly.
#Solved structures are deliberately NOT part of the height calculation.  The dataset is a
#prediction: AlphaFold models, Pfam domains, disorder and the sequence fallback.  Structures
#are merged in later as columns and used to validate it in S2 -- keeping them out is what makes
#that a comparison rather than a circularity.  They are also the smaller set: 393 ectodomains
#have one against 2,786 with a model, and on the 389 with both the model's confident span
#covers a median 98% of the ectodomain against 92% of residues resolved in the structure.

#combine homology and domain annotated dataframes to incoorporate all data
dom_plus_alphafold = pd.concat([heights,alphafold_measured], sort = False)

#remove unlabeled proteins (clean)
dom_plus_af = dom_plus_alphafold.dropna(subset = ['start', 'end', 'ID link'])

#insure that all proteins have an id assignment
dom_plus_af = dom_plus_af.assign(ID = dom_plus_af['ID link'] + '[' + 
                    dom_plus_af['start'].map(int).map(str) + ':' +
                    dom_plus_af['end'].map(int).map(str) +']')

#how many proteins/domains
dom_plus_af['ID'].describe()

In [ ]:
#### set score as max domain score for well modeled proteins
#will select alphafold models over domains in those cases
dom_plus_af = dom_plus_af.assign(score_dom = dom_plus_af.apply(
    lambda row: (dom_plus_af['score_dom'].max()+1) if (row['mean_confidence_trimmed'] >= 70) else row['score_dom'],
    axis = 1
))

#assign low score to alphafold models that have a confidence less than 70
#will choose domains in that case
dom_plus_af = dom_plus_af.assign(score_dom = dom_plus_af.apply(
    lambda row: -1.0 if (row['mean_confidence_trimmed'] < 70) else row['score_dom'], axis = 1
))

#how many still have no annotation?
#dom_plus_af[dom_plus_af['score_dom'].isnull()].sort_values('ID')

#find the overlaps between domains and alphafold
#filter out alphafold domains that are not valid or if the domain start/end do not conform
#since I filter valid here, no need to filter again later
dom_plus_af_len = dom_plus_af[~(dom_plus_af['mean_confidence_trimmed'] < 70) & (dom_plus_af['dom_start'] < dom_plus_af['dom_end'])].copy()

#Clip disordered regions off the modelled span before resolving overlaps.
#The AlphaFold row carries the maximum score_dom, so it wins its whole subgraph and any
#disorder region touching it is discarded outright -- its residues then fall through to the
#0.04 nm/aa sequence estimate.  And the tolerance is not the `overlap = 1` below: find_overlaps
#wraps its interval construction in a bare try/except and degrades the tolerance for the entire
#frame if any single row has end - start <= 2*overlap, which a two-residue multipass loop does,
#so a SINGLE residue of overlap is enough.  NPFF2 gains two residues of annotation between
#MobiDB 2019 and 2026, crosses the boundary by two, and loses a 132-residue polymer
#contribution: 21.0 nm to 8.6 nm.  Trimming makes a region overlapping by k residues lose k
#residues instead of all of them.
from surfaceomeTopography.domain_overlap import trim_to_unmodelled

dom_plus_af_len = trim_to_unmodelled(dom_plus_af_len)

alphafold_assigned =create_assignments(dom_plus_af_len, filter_field = 'dom', filter_score = True, drop_len = 1.0, overlap = 1)

# alphafold_overlaps = find_overlaps(dom_plus_af_len, allowed_overlap = 4, columns = ('dom_start','dom_end'))
# alphafold_graph = overlap_graph(alphafold_overlaps, grouping_var = 'ID link')
# alphafold_conns = find_connections(alphafold_graph)
# alphafold_data = annotate_subgraphs(alphafold_conns, alphafold_overlaps)

# alphafold_selected = alphafold_data.groupby(['ID link', 'Subgraph'])['score_dom', 'ID'].apply(select_max, col = 'score_dom')#domain_overlap.select_max, col = 'score_dom')
# alphafold_selected = alphafold_selected[alphafold_selected >= 0]

# alphafold_assigned = alphafold_overlaps.loc[alphafold_overlaps.ID.isin(alphafold_selected), :].drop(columns=['ID','Interval'])

# Import and incorporate structure measurements

while homology modeling and alpahfold should structurally sound predictions, an experimental structure is always best!
Here I import and incorporate structures from the PDB that have been mapped to surfacome protein extracelluar domains
(see S2_Methods_vs_Structural_Measurements.ipynb)

# Create combined homology/domain count/structure estimates

In [ ]:
##recreate id column
homology_assigned = homology_assigned.assign(ID = homology_assigned['ID link'] + '[' + 
                   homology_assigned['start'].map(int).map(str) + ':' +
                   homology_assigned['end'].map(int).map(str) +']')

##record homology model start/end
homology_assigned = homology_assigned.assign(homology_start = np.where(homology_assigned['Job id'].notnull(), 
                                                                       np.where(homology_assigned.valid, homology_assigned.dom_start, np.nan), np.nan))

homology_assigned = homology_assigned.assign(homology_end = np.where(homology_assigned['Job id'].notnull(), 
                                                                       np.where(homology_assigned.valid, homology_assigned.dom_end, np.nan), np.nan))

##record stuctured domain estimate start/end
homology_assigned = homology_assigned.assign(domain_start = np.where(homology_assigned['Pfam'].notnull(), homology_assigned.dom_start, np.nan))
homology_assigned = homology_assigned.assign(domain_end = np.where(homology_assigned['Pfam'].notnull(), homology_assigned.dom_end, np.nan))

#create domain length estimate
homology_assigned = homology_assigned.assign(domain_len =  homology_assigned.domain_end-homology_assigned.domain_start)


##disorder start/end already recorded


#cleanup assignments by dropping duplicates
homology_assigned = homology_assigned.drop_duplicates(subset =['ID', 'height','dom_start', 'dom_end', 'seq_len', 'source','dom_len', 'Hit', 'Confidence (%)', 'Sequence identity (%)',
       'Qstart', 'Qend', 'Hit info 1', 'Hit info 2', 'Hit info 3', 'ecd_len'])

#filter homology assignments that are not valid/have no measurements... 
homology_assigned = homology_assigned[~(homology_assigned.homology_start.notnull()&homology_assigned.valid.isnull())]
#create domain type annotion (homology model, structured domain, disorder domain, )

##create combined homology/domain height estimates
#assign homology model height in nm (rather than angstrom)


# #import surfaceome mapping to gene names
# names = pd.read_csv(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab'),
#                       sep = '\t')
#merge with names
assigned_named = pd.merge(homology_assigned, names[['Entry name', 'Entry']], left_on = 'ID link', right_on = 'Entry', 
                          how = 'left',
                          #indicator = True
                         )
#empty string for na
assigned_named.loc[:,'Entry name'].fillna("", inplace = True)

#first scale max_dim to nm
assigned_named['height'] = assigned_named.apply(
    lambda row: row['max_dim']/10 if (np.isnan(row['max_dim']) == False) else row['height'],
    axis = 1
)

#assign plexins to min height
assigned_named['height'] = assigned_named.apply(
    lambda row: row['min_dim']/10 if 'PLX' in row['Entry name'] else row['height'],
    axis = 1
)


##annotate combined estimates and homology only, disorder, and domain
#create an annotation column, groupby and list all unique categories?
assigned_named = assigned_named.assign(methods = np.where(assigned_named.homology_start.notnull(), 'homology', np.where(assigned_named.disorder_start.notnull(), 'disorder', 'domain')))

assigned_named.to_csv(out_path('database/domain_disorder_assignments_homology.csv'))

In [ ]:
#create size and coverage estimates using domain counting, disorder annotations, and homology modeling

heights_homo = assigned_named.groupby('ID').agg(   
    {  
        'ID link': 'first',
        'Entry name': 'first',
        'height':['count','sum'],    # Sum height
        'max_dim': 'first', #testing
        'min_dim': 'first', #testing
        'Job id': 'first', #job id for visualization
        'Hit': 'first',
        'valid': 'all',
         'disorder_len': ['count','sum'],
         'dom_len': ['count','sum'],
        'qlen': ['count','sum'],
        'homology_len': ['sum','count'],
        'domain_len': ['sum','count'],
        'Confidence (%)': 'first',
        'Sequence identity (%)': 'first',
        'seq_len': 'first',
        'source': 'last',
        'methods': lambda x: ','.join(sorted(pd.Series.unique(x))),
        'disorder_start': 'min',
        'disorder_end': 'max',
        'domain_start':'min',
        'domain_end': 'max',
        'homology_start': 'min',
        'homology_end': 'max',
    })

heights_homo.columns = ["_".join(x) for x in heights_homo.columns.ravel()]
heights_homo.reset_index(inplace = True)


#coverage of the ectodomain by its annotations, percent
heights_homo = annotation_coverage(heights_homo, 'homology_len_sum')

#create full estimates with domains/homology, filter out incompletely and poorly modeled proteins
#all homology models have been filtered for validitity and confidence above (doing this as it is smoothes out some filtering)
well_modeled = heights_homo[(heights_homo['best_coverage'] >= 70)]

#this is currently a list of modeled ECDs and not filtered for each unique protein
#need to select the representative ECD for each protein ultimately
#leave this for last as different selection methods could be used?

#number of well modeled for each type of modeling
well_modeled['methods_<lambda>'].value_counts()

# Create combined Alphafold/domain count estimates

In [ ]:
##recreate id column
alphafold_assigned = alphafold_assigned.assign(ID = alphafold_assigned['ID link'] + '[' + 
                   alphafold_assigned['start'].map(int).map(str) + ':' +
                   alphafold_assigned['end'].map(int).map(str) +']')

##record alphafold model start/end
alphafold_assigned = alphafold_assigned.assign(alphafold_start = np.where(alphafold_assigned.mean_confidence_trimmed.notnull(),alphafold_assigned.dom_start, np.nan))

alphafold_assigned = alphafold_assigned.assign(alphafold_end = np.where(alphafold_assigned.mean_confidence_trimmed.notnull(),alphafold_assigned.dom_end, np.nan))

##record stuctured domain estimate start/end
alphafold_assigned = alphafold_assigned.assign(domain_start = np.where(alphafold_assigned['Pfam'].notnull(), alphafold_assigned.dom_start, np.nan))
alphafold_assigned = alphafold_assigned.assign(domain_end = np.where(alphafold_assigned['Pfam'].notnull(), alphafold_assigned.dom_end, np.nan))

#create domain length estimate
alphafold_assigned = alphafold_assigned.assign(domain_len =  alphafold_assigned.domain_end-alphafold_assigned.domain_start)


##disorder start/end already recorded


#cleanup assignments by dropping duplicates
alphafold_assigned = alphafold_assigned.drop_duplicates(subset =['ID', 'height','dom_start', 'dom_end', 'seq_len', 'source','dom_len', 'ecd_length'])

#filter alphafold assignments that are not valid/have no measurements... 
#alphafold_assigned = alphafold_assigned[(alphafold_assigned.mean_confidence_trimmed.isnull())]
#create domain type annotion (alphafold model, structured domain, disorder domain, )

##create combined alphafold/domain height estimates
#assign alphafold model height in nm (rather than angstrom)


#merge with names
af_assigned_named = pd.merge(alphafold_assigned, names[['Entry name', 'Entry']], left_on = 'ID link', right_on = 'Entry', 
                          how = 'left',
                          #indicator = True
                         )
#empty string for na
af_assigned_named.loc[:,'Entry name'].fillna("", inplace = True)

#first scale max_dim to nm
af_assigned_named['height'] = af_assigned_named.apply(
    lambda row: row['max_dim']/10 if (np.isnan(row['max_dim']) == False) else row['height'],
    axis = 1
)

#assign plexins to min height
af_assigned_named['height'] = af_assigned_named.apply(
    lambda row: row['min_dim']/10 if 'PLX' in row['Entry name'] else row['height'],
    axis = 1
)


##annotate combined estimates and alphafold only, disorder, and domain
#create an annotation column, groupby and list all unique categories?
af_assigned_named = af_assigned_named.assign(methods = np.where(af_assigned_named.mean_confidence_trimmed.notnull(), 'alphafold', np.where(af_assigned_named.disorder_start.notnull(), 'disorder', 'domain')))


af_assigned_named = af_assigned_named.assign(seq_len = np.where(af_assigned_named.seq_len.isnull(), af_assigned_named.ecd_length, af_assigned_named.seq_len))


af_assigned_named.to_csv(out_path('database/domain_disorder_assignments_alphafold.csv'))

In [ ]:
#create size and coverage estimates using domain counting, disorder annotations, and alphafold modeling

heights_alpha = af_assigned_named.groupby('ID').agg(   
    {  
        'ID link': 'first',
        'Entry name': 'first',
        'height':['count','sum'],    # Sum height
        'max_dim': 'first', #testing
        'min_dim': 'first', #testing
        'af_ID': 'first',
         'disorder_len': ['count','sum'],
         'dom_len': ['count','sum'],
        'qlen': ['count','sum'],
        'alphafold_len': ['sum','count'],
        'domain_len': ['sum','count'],
        'mean_confidence_trimmed': 'first',
        'seq_len': 'first',
        'source': 'last',
        'methods': lambda x: ','.join(sorted(pd.Series.unique(x))),
        'disorder_start': 'min',
        'disorder_end': 'max',
        'domain_start':'min',
        'domain_end': 'max',
        'alphafold_start': 'min',
        'alphafold_end': 'max',
    })

heights_alpha.columns = ["_".join(x) for x in heights_alpha.columns.ravel()]
heights_alpha.reset_index(inplace = True)



#coverage of the ectodomain by its annotations, percent
heights_alpha = annotation_coverage(heights_alpha, 'alphafold_len_sum')

#create full estimates with domains/alphafold, filter out incompletely and poorly modeled proteins
#all alphafold models have been filtered for validitity and confidence above (doing this as it is smoothes out some filtering)
af_well_modeled = heights_alpha[(heights_alpha['best_coverage'] >= 70)]

#this is currently a list of modeled ECDs and not filtered for each unique protein
#need to select the representative ECD for each protein ultimately
#leave this for last as different selection methods could be used?

#number of well modeled for each type of modeling
af_well_modeled['methods_<lambda>'].value_counts()

In [ ]:
import seaborn as sns
sns.distplot(well_modeled['seq_len_first'])

In [ ]:
#assign ID column for ECDs
ecds_classified = ecds_classified.assign(ID = ecds_classified['ID link'] + '[' + 
                   ecds_classified['start'].map(int).map(str) + ':' +
                   ecds_classified['end'].map(int).map(str) +']')

# Alphafold model QC

In [ ]:
##add back any ECD domains that fell out of the QC (short)/otherwise missing assignment
#recover proteins marked as surfaceome, but otherwise not measured

af_well_modeled_classified = ecds_classified[ecds_classified['ID'].isin(af_well_modeled.ID)].drop_duplicates(subset = ['ID'])
unmodled_classified = ecds_classified[~ecds_classified['ID'].isin(af_well_modeled.ID)].drop_duplicates(subset = ['ID'])

##create sequence length estimates for any missing protiens
#supplement incompletely modeled proteins with a sequence length estimate

#gather domains/alphafold models of incompletely modeled proteins
incomplete_models = af_assigned_named[~af_assigned_named.ID.isin(af_well_modeled.ID)]

#complete modeles for later
complete_models = af_assigned_named[af_assigned_named.ID.isin(af_well_modeled.ID)]

incomplete_models.to_csv(out_path('database/incomplete_models_alphafold.csv'))
complete_models.to_csv(out_path('database/complete_models_alphafold.csv'))

#groupby ECD domain (ID) and create


#incomplete_models.seq_len - incomplete_models.domain_len - incomplete_models.alphafold_len

#create sum of each domain type
incomplete_sum = incomplete_models.groupby('ID')[['alphafold_len', 'domain_len', 'disorder_len']].sum()

#record first/last info
incomplete_min = incomplete_models.groupby('ID')[['alphafold_start', 'domain_start', 'disorder_start']].min()
incomplete_max = incomplete_models.groupby('ID')[['alphafold_end', 'domain_end', 'disorder_end']].max()
incomplete_first = incomplete_models.groupby('ID')[['af_ID','ID link','Entry name']].first()

In [ ]:
incomplete_heights = incomplete_models.groupby('ID').agg(   
    {  
        'ID link': 'first',
        'Entry name': 'first',
        'height':['count','sum'],    # Sum height
        'max_dim': 'first', #testing
        'min_dim': 'first', #testing
        'af_ID': 'first', #job id for visualization
        'disorder_len': ['count','sum'],
        'dom_len': ['count','sum'],
        'qlen': ['count','sum'],
        'mean_confidence_trimmed': 'first',
        'alphafold_len': ['sum','count'],
        'domain_len': ['sum','count'],
        'seq_len': 'first',
        'source': 'last',
        'methods': lambda x: ','.join(sorted(pd.Series.unique(x))),
        'disorder_start': 'min',
        'disorder_end': 'max',
        'domain_start':'min',
        'domain_end': 'max',
        'alphafold_start': 'min',
        'alphafold_end': 'max',
        
        
    })

incomplete_heights.columns = ["_".join(x) for x in incomplete_heights.columns.ravel()]
incomplete_heights.reset_index(inplace = True)


#coverage of the ectodomain by its annotations, percent
incomplete_heights = annotation_coverage(incomplete_heights, 'alphafold_len_sum')

#find missing length of amino acids
incomplete_heights = incomplete_heights.assign(missing_len = incomplete_heights.seq_len_first\
                          - incomplete_heights.domain_len_sum\
                          - incomplete_heights.alphafold_len_sum\
                          - incomplete_heights.disorder_len_sum 
                         )

#ADD SEQUENCE TO METHODS CLASSIFICATION
incomplete_heights['methods_<lambda>'] = incomplete_heights['methods_<lambda>'] +',sequence'

#residual height estimate based on missing number of amino acids
#need to create a linear fit of structural models in order to do these estimates in a more justifiable way...
#filler for now
#should keep loop classifications etc
incomplete_heights = incomplete_heights.assign(sequence_height = incomplete_heights['missing_len']* 0.04)


#add to previous height sum for total height estimate
incomplete_heights = incomplete_heights.assign(imputed_height = incomplete_heights.height_sum + incomplete_heights.sequence_height)

#add total height columns
incomplete_heights = incomplete_heights.assign(total_height = incomplete_heights.imputed_height)


#id ECDs still missing
unmodeled_ecds = unmodled_classified[~unmodled_classified.ID.isin(incomplete_heights.ID)]

#create sequence length height estimate (see above, filler for now, may need to distingish loops)
#SEQ LEN ESTIMATE NEED TO JUSTIFY!! 6nm per 150 amino acids, current...
unmodeled_ecds = unmodeled_ecds.assign(unmodeled_height = unmodeled_ecds['seq_len']*0.04)

unmodeled_ecds = unmodeled_ecds.assign(total_height = unmodeled_ecds.unmodeled_height)

unmodeled_ecds['methods_<lambda>'] = 'sequence'

#conform to other height totals
unmodeled_ecds.rename(columns = {
    'ID link' : 'ID link_first',
     'Entry name': 'Entry name_first',
    'seq_len': 'seq_len_first',
    'source': 'source_last',
    'start': 'seq_start',
    'end':'seq_end'
    }, inplace = True)

#select representative ECD
#use longest sequence length as the respresentative ECD
#MAYBE CHANGE? are there any instences where this is not appropriate?

# def largest_unique_ecd(df, groupby_col, selection_col):
#     #select for the largest ecd when there are multiple measured ecds
    
#     idx = df.groupby([groupby_col])[selection_col].transform(max) == df[selection_col]
    
#     return df[idx].drop_duplicates(subset = [groupby_col])

unmodeled_longest = largest_unique_ecd(unmodeled_ecds[unmodeled_ecds['methods_<lambda>'] == 'sequence'], 'ID link_first', 'seq_len_first')
unmodeled_longest.to_csv(out_path('database/unmodeled_ecds_longest_seq.csv'))

unmodeled_longest['ID link_first'].describe()

af_well_modeled = af_well_modeled.assign(total_height = af_well_modeled.height_sum)

#combine all estimates to come up with estimates for all ECDs
all_estimates = pd.concat([unmodeled_ecds[['ID', 'ID link_first', 'seq_start', 'seq_end', 'seq_len_first', 'unmodeled_height', 'total_height', 'methods_<lambda>']], incomplete_heights,
           af_well_modeled]).reset_index()

longest_seq_estimates_alphafold = largest_unique_ecd(all_estimates, 'ID link_first', 'seq_len_first')

longest_seq_estimates_alphafold.to_csv(out_path('database/height_estimates_alphafold.csv'), index = False)

# Homology modeling QC

In [ ]:
##add back any ECD domains that fell out of the QC (short)/otherwise missing assignment
#recover proteins marked as surfaceome, but otherwise not measured

#assign ID column for ECDs

well_modeled_classified = ecds_classified[ecds_classified['ID'].isin(well_modeled.ID)].drop_duplicates(subset = ['ID'])
unmodled_classified = ecds_classified[~ecds_classified['ID'].isin(well_modeled.ID)].drop_duplicates(subset = ['ID'])

##create sequence length estimates for any missing protiens
#supplement incompletely modeled proteins with a sequence length estimate

#gather domains/homology models of incompletely modeled proteins
incomplete_models = assigned_named[~assigned_named.ID.isin(well_modeled.ID)]

#complete modeles for later
complete_models = assigned_named[assigned_named.ID.isin(well_modeled.ID)]

incomplete_models.to_csv(out_path('database/incomplete_models_homology.csv'))
complete_models.to_csv(out_path('database/complete_models_homology.csv'))

#groupby ECD domain (ID) and create


#incomplete_models.seq_len - incomplete_models.domain_len - incomplete_models.homology_len

#create sum of each domain type
incomplete_sum = incomplete_models.groupby('ID')[['homology_len', 'domain_len', 'disorder_len']].sum()

#record first/last info
incomplete_min = incomplete_models.groupby('ID')[['homology_start', 'domain_start', 'disorder_start']].min()
incomplete_max = incomplete_models.groupby('ID')[['homology_end', 'domain_end', 'disorder_end']].max()
incomplete_first = incomplete_models.groupby('ID')[['Job id','ID link','Entry name']].first()

incomplete_heights = incomplete_models.groupby('ID').agg(   
    {  
        'ID link': 'first',
        'Entry name': 'first',
        'height':['count','sum'],    # Sum height
        'max_dim': 'first', #testing
        'min_dim': 'first', #testing
        'Job id': 'first', #job id for visualization
        'Hit': 'first',
        'valid': 'all',
        'disorder_len': ['count','sum'],
        'dom_len': ['count','sum'],
        'qlen': ['count','sum'],
        'homology_len': ['sum','count'],
        'domain_len': ['sum','count'],
        'Confidence (%)': 'first',
        'Sequence identity (%)': 'first',
        'seq_len': 'first',
        'source': 'last',
        'methods': lambda x: ','.join(sorted(pd.Series.unique(x))),
        'disorder_start': 'min',
        'disorder_end': 'max',
        'domain_start':'min',
        'domain_end': 'max',
        'homology_start': 'min',
        'homology_end': 'max',
        
        
    })

incomplete_heights.columns = ["_".join(x) for x in incomplete_heights.columns.ravel()]
incomplete_heights.reset_index(inplace = True)


#coverage of the ectodomain by its annotations, percent
incomplete_heights = annotation_coverage(incomplete_heights, 'homology_len_sum')

#find missing length of amino acids
incomplete_heights = incomplete_heights.assign(missing_len = incomplete_heights.seq_len_first\
                          - incomplete_heights.domain_len_sum\
                          - incomplete_heights.homology_len_sum\
                          - incomplete_heights.disorder_len_sum 
                         )

#ADD SEQUENCE TO METHODS CLASSIFICATION
incomplete_heights['methods_<lambda>'] = incomplete_heights['methods_<lambda>'] +',sequence'

#residual height estimate based on missing number of amino acids
#need to create a linear fit of structural models in order to do these estimates in a more justifiable way...
#filler for now
#should keep loop classifications etc
incomplete_heights = incomplete_heights.assign(sequence_height = incomplete_heights['missing_len']* 0.04)


#add to previous height sum for total height estimate
incomplete_heights = incomplete_heights.assign(imputed_height = incomplete_heights.height_sum + incomplete_heights.sequence_height)

#add total height columns
incomplete_heights = incomplete_heights.assign(total_height = incomplete_heights.imputed_height)


#id ECDs still missing
unmodeled_ecds = unmodled_classified[~unmodled_classified.ID.isin(incomplete_heights.ID)]

#create sequence length height estimate (see above, filler for now, may need to distingish loops)
#SEQ LEN ESTIMATE NEED TO JUSTIFY!! 6nm per 150 amino acids, current...
unmodeled_ecds = unmodeled_ecds.assign(unmodeled_height = unmodeled_ecds['seq_len']*0.04)

unmodeled_ecds = unmodeled_ecds.assign(total_height = unmodeled_ecds.unmodeled_height)

unmodeled_ecds['methods_<lambda>'] = 'sequence'

#conform to other height totals
unmodeled_ecds.rename(columns = {
    'ID link' : 'ID link_first',
    'seq_len': 'seq_len_first',
    'source': 'source_last',
    'start': 'seq_start',
    'end':'seq_end'
    }, inplace = True)

#select representative ECD
#use longest sequence length as the respresentative ECD
#MAYBE CHANGE? are there any instences where this is not appropriate?

# def largest_unique_ecd(df, groupby_col, selection_col):
#     #select for the largest ecd when there are multiple measured ecds
    
#     idx = df.groupby([groupby_col])[selection_col].transform(max) == df[selection_col]
    
#     return df[idx].drop_duplicates(subset = [groupby_col])

unmodeled_longest = largest_unique_ecd(unmodeled_ecds[unmodeled_ecds['methods_<lambda>'] == 'sequence'], 'ID link_first', 'seq_len_first')
unmodeled_longest.to_csv(out_path('database/unmodeled_ecds_longest_seq.csv'))

unmodeled_longest['ID link_first'].describe()

well_modeled = well_modeled.assign(total_height = well_modeled.height_sum)

#combine all estimates to come up with estimates for all ECDs
all_estimates = pd.concat([unmodeled_ecds[['ID', 'ID link_first', 'seq_start', 'seq_end', 'seq_len_first', 'unmodeled_height', 'total_height', 'methods_<lambda>']], incomplete_heights,
           well_modeled]).reset_index()

longest_seq_estimates_homology = largest_unique_ecd(all_estimates, 'ID link_first', 'seq_len_first')

longest_seq_estimates_homology.to_csv(out_path('database/height_estimates_homology.csv'), index = False)

In [ ]:
#The height table is a prediction. Solved structures are compared against it in S2 rather than
#merged into it here -- and S2 is also what produces the structure table, after this notebook, so
#reading it here meant reading a stored copy from the run before.
alphafold_structures_all = longest_seq_estimates_alphafold.copy()
alphafold_structures_all = alphafold_structures_all.assign(methods = alphafold_structures_all['methods_<lambda>'])

#merge with names
alphafold_structures_all = pd.merge(alphafold_structures_all, names[['ID link','Entry name']], left_on = 'ID link_first', right_on = 'ID link', 
                          how = 'left',
                          #indicator = True
                         )

alphafold_structures_all['Entry name_first'] = alphafold_structures_all['Entry name']
alphafold_structures_all.drop(columns = ['Entry name', 'ID link'], inplace = True)

In [ ]:
#classification of every surfaceome protein: UniProt location and topology, the SURFY surface
#call, Almen functional class, CSPA evidence (build_surfaceome_classification.py). The
#surfaceome stays inclusive; a stricter surface set is a filter on these columns.
classification = pd.read_csv(resolve('inputs/annotation/surfaceome_classification.csv'))
alphafold_structures_all = alphafold_structures_all.merge(classification, left_on = 'ID link_first',
                                                          right_on = 'accession', how = 'left').drop(columns = 'accession')

alphafold_structures_all.to_csv(out_path('database/height_estimates.csv'), index = False)

In [ ]:
#VISUAL OF FULL DATASET
#histogram on linear scale

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

fig = plt.figure(figsize=(10,7), constrained_layout=True)

df = longest_seq_estimates_homology[~longest_seq_estimates_homology ['ID link_first'].duplicated()]

well = df.total_height.notnull()

x = df.loc[well]['total_height']
y = df.loc[well]['total_height']
#z = df.loc[well & (df['multipass'] == True)]['best_height']
#a = df.loc[well & (df['multipass'] == False)& (df['source_y'] != 'gpi')]['best_height']
#b = df.loc[well & (df['multipass'] == False)& (df['source_y'] == 'gpi')]['best_height']

gs = fig.add_gridspec(5, 1, figure = fig)

ax1 = fig.add_subplot(gs[0:1, 0])
ax1.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#00A8A8')
#ax1.plot(z, np.zeros(z.shape), 'b|', ms=20, color = '#CD489A')
#ax1.plot(a, np.zeros(a.shape), 'b|', ms=20, color = '#454545')# rug plot
#ax1.plot(b, np.zeros(b.shape), 'b|', ms=20, color = '#00A8A8')
#ax1.set_ylim([-0.01, 0.01])
#hist, bins, _ = plt.hist(x, bins=100)
#plt.yscale('log')
plt.xlabel('height (nm)')
#plt.xscale('log')
#plt.ylabel('log count')
#bins = range(1,1000,10)

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(0.2),np.log10(800),100)


ax2 = fig.add_subplot(gs[1:4, 0])
plt.hist(y, bins=logbins, histtype = 'step', color = '#00A8A8')
#plt.hist(z, bins=logbins, histtype = 'step', color = '#CD489A')
#plt.hist(a, bins=logbins, histtype = 'step', color = '#454545')
#plt.hist(b, bins=logbins, histtype = 'step', color = '#454545')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax2.get_xticklabels(), visible=True)

plt.xlabel('height (nm)')
plt.legend(['all', 'multipass', 'singlepass'])

#ax2 = fig.add_subplot(gs[4:5, 0])

#plt.hist(x, bins=logbins, histtype = 'step', color = '#454545')
#plt.xscale('log')
#plt.xlabel('log height (nm)')
#plt.ylabel('count')
#plt.setp(ax2.get_xticklabels(), visible=True)

#plt.xlabel('height (nm)')
#plt.legend(['gpi'])

#plt.xscale('log')

plt.savefig(out_path('figures/01_heights_all_proteins_homology.pdf'))

In [ ]:
#VISUAL OF FULL DATASET
#histogram on linear scale

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

fig = plt.figure(figsize=(10,5), constrained_layout=True)

df = alphafold_structures_all[~alphafold_structures_all['ID link_first'].duplicated()]

well = df.total_height.notnull()

x = df.loc[well]['total_height']
y = df.loc[well]['total_height']
#z = df.loc[well & (df['multipass'] == True)]['best_height']
#a = df.loc[well & (df['multipass'] == False)& (df['source_y'] != 'gpi')]['best_height']
#b = df.loc[well & (df['multipass'] == False)& (df['source_y'] == 'gpi')]['best_height']

gs = fig.add_gridspec(5, 1, figure = fig)

ax1 = fig.add_subplot(gs[0:1, 0])
ax1.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#00A8A8')
#ax1.plot(z, np.zeros(z.shape), 'b|', ms=20, color = '#CD489A')
#ax1.plot(a, np.zeros(a.shape), 'b|', ms=20, color = '#454545')# rug plot
#ax1.plot(b, np.zeros(b.shape), 'b|', ms=20, color = '#00A8A8')
#ax1.set_ylim([-0.01, 0.01])
#hist, bins, _ = plt.hist(x, bins=100)
#plt.yscale('log')
plt.xlabel('height (nm)')
#plt.xscale('log')
#plt.ylabel('log count')
#bins = range(1,1000,10)

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(0.2),np.log10(800),100)


ax2 = fig.add_subplot(gs[1:4, 0])
plt.hist(y, bins=logbins, histtype = 'step', color = '#00A8A8')
#plt.hist(z, bins=logbins, histtype = 'step', color = '#CD489A')
#plt.hist(a, bins=logbins, histtype = 'step', color = '#454545')
#plt.hist(b, bins=logbins, histtype = 'step', color = '#454545')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax2.get_xticklabels(), visible=True)

plt.xlabel('height (nm)')
plt.legend(['all', 'multipass', 'singlepass'])

#ax2 = fig.add_subplot(gs[4:5, 0])

#plt.hist(x, bins=logbins, histtype = 'step', color = '#454545')
#plt.xscale('log')
#plt.xlabel('log height (nm)')
#plt.ylabel('count')
#plt.setp(ax2.get_xticklabels(), visible=True)

#plt.xlabel('height (nm)')
#plt.legend(['gpi'])

#plt.xscale('log')

plt.savefig(out_path('figures/01_heights_all_proteins_alphafold.pdf'))

# Domain counting only estimates

In [ ]:
heights_total = heights_total.rename(columns = {'target_name':'ID'})

#Domain-counting-only heights -- Supplementary Table 5, and S2's domain-counting comparison. A
#separate notebook used to rebuild the whole surfaceome to produce this, with the older on/off
#disorder model; this is the same aggregation, under the current model, written once.
heights_total.to_csv(out_path('database/height_estimates_domain_counting.csv'), index = False)

In [ ]:
#merge homology modeling calculated heights and domain counting calculated heights
#h_dom_counts = pd.merge(longest_seq_estimates_alphafold, heights_total, on = ['ID', 'ID link_first'], 
h_dom_counts = pd.merge(alphafold_structures_all, heights_total, on = ['ID', 'ID link_first'], 
                        suffixes = ('_homology', '_domain'), how = 'left')

In [ ]:
h_dom_counts.to_csv(out_path('database/height_estimates_with_domain_counts.csv'))

In [ ]:
#DOMAIN COUNTING AND DISORDER BREAKDOWN

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


# histogram on linear scale
#plt.figure(figsize=(10,10))

fig = plt.figure(figsize=(10,8), constrained_layout=True)
gs = fig.add_gridspec(7, 1, figure = fig)

ax1 = fig.add_subplot(gs[0:1, 0])


df = h_dom_counts

x = df.loc[ df.total_height > 0]['total_height']
y = df.loc[(df['disorder_len_count_domain'] > 0) &(df['alphafold_len_count'] == 0)]['total_height']

z = df.loc[ ((df['disorder_len_count_domain'] > 0)|(df['disorder_len_count_homology'] > 0))]['total_height']
a = df.loc[ ((df['disorder_len_count_domain'] > 0)|(df['disorder_len_count_homology'] > 0)) &((df['alphafold_len_count'] == 0)|(df['height_count_homology'] == 0))]['total_height']
b = df.loc[ ((df['disorder_len_count_domain'] > 0)|(df['disorder_len_count_homology'] > 0)) &((df['dom_len_count_domain'] > 0)|(df['alphafold_len_count'] > 0))]['total_height']
c = df.loc[((df['dom_len_count_domain'] > 0)|(df['dom_len_count_homology'] >0))&((df['disorder_len_count_domain'] == 0)&(df['disorder_len_count_homology'] == 0))]['total_height']
#d = df.loc[ (df['dom_len_count_domain'] >= 5)]['total_height']
#e = df.loc[well & (df['dom_len_count_domain'] >= 6)]['total_height']
#f = df.loc[well & (df['dom_len_count_domain'] > 6)]['total_height']


ax1.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#00A8A8')  # rug plot
#ax1.set_ylim([-0.01, 0.01])
#hist, bins, _ = plt.hist(x, bins=100)
#plt.yscale('log')
plt.xlabel('height (nm)')
#plt.ylabel('log count')
#bins = range(1,1000,10)

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(0.2),np.log10(900),90)


ax2 = fig.add_subplot(gs[1:2, 0])
plt.hist(z, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax2.get_xticklabels(), visible=False)

ax3 = fig.add_subplot(gs[2:3, 0], sharex = ax2, sharey = ax2)
plt.hist(a, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
plt.ylabel('count')
ax3.plot(a, [0.01]*len(a), '|')
plt.setp(ax3.get_xticklabels(), visible=False)

# ax4 = fig.add_subplot(gs[3:4, 0], sharex = ax2, sharey = ax2)
# plt.hist(b, bins=logbins, histtype = 'step', color = '#00A8A8')
# plt.xscale('log')
# #plt.xlabel('log height (nm)')
# plt.ylabel('count')
# plt.setp(ax4.get_xticklabels(), visible=False)

# ax5 = fig.add_subplot(gs[4:5, 0], sharex = ax2, sharey = ax2)
# plt.hist(c, bins=logbins, histtype = 'step', color = '#00A8A8')
# plt.xscale('log')
# #plt.xlabel('log height (nm)')
# plt.ylabel('count')
# plt.setp(ax5.get_xticklabels(), visible=False)

# ax6 = fig.add_subplot(gs[5:6, 0], sharex = ax2, 
#                       #sharey = ax2
#                      )
# plt.hist(d, bins=logbins, histtype = 'step', color = '#00A8A8')
# plt.xscale('log')
# #plt.xlabel('log height (nm)')
# plt.ylabel('count')
# plt.setp(ax6.get_xticklabels(), visible=False)

# ax7 = fig.add_subplot(gs[6:7, 0], sharex = ax2, 
#                       #sharey = ax2
#                      )
# plt.hist(y, bins=logbins, histtype = 'step', color = '#00A8A8')
# plt.xscale('log')
# plt.xlabel('log height (nm)')
# plt.ylabel('count')
# #plt.setp(ax6.get_xticklabels(), visible=False)


#ax2.plot(x, np.zeros(x.shape), 'b|', ms=20, alpha = 0.5, color = '#00A8A8')

plt.savefig(out_path('figures/01_heights_by_method_and_disorder.pdf'),
           transparent = True)

In [ ]:
#DOMAIN COUNTING AND DISORDER BREAKDOWN

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


# histogram on linear scale
#plt.figure(figsize=(10,10))

fig = plt.figure(figsize=(10,8), constrained_layout=True)
gs = fig.add_gridspec(7, 1, figure = fig)

ax1 = fig.add_subplot(gs[0:1, 0])

well = h_dom_counts.dom_coverage_domain >= 0.5
df = h_dom_counts

x = df.loc[well]['total_height']
y = df.loc[well &(df['dom_len_count_domain'] == 0)]['total_height']
z = df.loc[well &(df['dom_len_count_domain'] == 1)]['total_height']
a = df.loc[well &(df['dom_len_count_domain'] == 2) ]['total_height']
b = df.loc[well &(df['dom_len_count_domain'] == 3)]['total_height']
c = df.loc[well &(df['dom_len_count_domain'] == 4)]['total_height']
d = df.loc[well &(df['dom_len_count_domain'] >= 5)]['total_height']
#e = df.loc[(df['dom_len_count_domain']  6)]['total_height']
#f = df.loc[(df['dom_len_count_domain'] > 6)]['total_height']
#e = df.loc[well & (df['dom_len_count_domain'] >= 6)]['total_height']
#f = df.loc[well & (df['dom_len_count_domain'] > 6)]['total_height']


ax1.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#00A8A8')  # rug plot
#ax1.set_ylim([-0.01, 0.01])
#hist, bins, _ = plt.hist(x, bins=100)
#plt.yscale('log')
plt.xlabel('height (nm)')
#plt.ylabel('log count')
#bins = range(1,1000,10)

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(0.2),np.log10(900),90)


ax2 = fig.add_subplot(gs[1:2, 0])
plt.hist(z, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax2.get_xticklabels(), visible=False)

ax3 = fig.add_subplot(gs[2:3, 0], sharex = ax2, 
                      sharey = ax2
                     )
plt.hist(a, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
plt.ylabel('count')
plt.setp(ax3.get_xticklabels(), visible=False)

ax4 = fig.add_subplot(gs[3:4, 0], sharex = ax2, sharey = ax2)
plt.hist(b, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax4.get_xticklabels(), visible=False)

ax5 = fig.add_subplot(gs[4:5, 0], sharex = ax2, sharey = ax2)
plt.hist(c, bins=logbins, histtype = 'step', color = '#00A8A8')
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax5.get_xticklabels(), visible=False)

ax6 = fig.add_subplot(gs[5:6, 0], sharex = ax3, 
                      sharey = ax2
                     )
plt.hist(d, bins=logbins, histtype = 'step', color = '#00A8A8',)
plt.xscale('log')
#plt.xlabel('log height (nm)')
plt.ylabel('count')
plt.setp(ax6.get_xticklabels(), visible=False)

# ax7 = fig.add_subplot(gs[6:7, 0], sharex = ax3, 
#                       sharey = ax2
#                      )
# plt.hist(y, bins=logbins, histtype = 'step', color = '#00A8A8')
# plt.xscale('log')
# plt.xlabel('log height (nm)')
# plt.ylabel('count')
# #plt.setp(ax6.get_xticklabels(), visible=False)


#ax2.plot(x, np.zeros(x.shape), 'b|', ms=20, alpha = 0.5, color = '#00A8A8')

plt.savefig(out_path('figures/01_heights_by_domain_count.pdf'),
           transparent = True)